<a href="https://colab.research.google.com/github/DaoPhang/Algo-Project/blob/main/Algo_Project_OCC8_Survivors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Clone the Datasets from Github

In [1]:
!git clone https://github.com/DaoPhang/Algo-Project.git
!pip install openpyxl tabulate --quiet

Cloning into 'Algo-Project'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 131 (delta 58), reused 51 (delta 19), pack-reused 12 (from 1)
Receiving objects: 100% (131/131), 53.45 MiB | 40.07 MiB/s, done.
Resolving deltas: 100% (64/64), done.


# Part 2 — The Double Agent Registry
## Algorithm Implementation (Set B)

## Step 1 — Install & Imports

In [1]:
import os
import time
import openpyxl
from collections import defaultdict
from itertools import combinations
from tabulate import tabulate

## Step 2 — Load Dataset

In [2]:
# ─────────────────────────────────────────────────────────────────────
# 1. Auto-detect Dataset Path
# ─────────────────────────────────────────────────────────────────────
import os
import zipfile
import openpyxl

EXCEL_PATH = None

for root, dirs, files in os.walk("/content/Algo-Project"):
    # skip macOS hidden metadata folder
    if "__MACOSX" in root:
        continue

    for file in files:
        # skip macOS hidden metadata files
        if file.startswith("._"):
            continue

        # find the real Part 2 Excel file
        if file.endswith(".xlsx") and "Part 2" in file:
            possible_path = os.path.join(root, file)

            # .xlsx files are actually zip-based, so this confirms it is real
            if zipfile.is_zipfile(possible_path):
                EXCEL_PATH = possible_path
                break

    if EXCEL_PATH is not None:
        break

if EXCEL_PATH is None:
    raise FileNotFoundError("Real Part 2.xlsx dataset not found. Check your repo folder.")

print(f"Dataset found: {EXCEL_PATH}")

SHEET_NAME = "B"
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

agents = []

for row in ws.iter_rows(min_row=3, values_only=True):
    if row[1] is None or row[1] == "Agent_ID":
        continue

    agents.append({
        "Agent_ID"        : row[1],
        "Alias"           : row[2],
        "Nationality_Code": row[3],
        "Last_Known_City" : row[4],
        "Access_Key"      : row[5],
        "Status"          : row[6],
        "Linked_Site"     : row[7],
    })

print(f"Loaded {len(agents)} agent records from Sheet '{SHEET_NAME}'\n")

Dataset found: C:\Users\Dao Phang\My Drive\GoodNotes\UM\Sem 3\WIA2005 Algo\Algo Project\Datasets\Part 2.xlsx
Loaded 14 agent records from Sheet 'B'



## Step 3 — Helper Function: Levenshtein Distance

In [3]:
# ─────────────────────────────────────────────────────────────────────
# 2. Helper — Levenshtein Distance
# ─────────────────────────────────────────────────────────────────────
def levenshtein(s1, s2):
    """
    Dynamic programming edit distance.
    Returns minimum number of single-character edits (insert, delete, substitute).
    Complexity: O(m x n) where m, n are string lengths.
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] if s1[i-1] == s2[j-1] \
                       else 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]

## Step 4 — Algorithm 1: Hash Table + Levenshtein Distance (Chosen)

In [4]:
# ─────────────────────────────────────────────────────────────────────
# 3. ALGORITHM 1 — Hash Table + Levenshtein Distance (CHOSEN)
#
#    Layer 1: Hash Table groups by access key, city, and alias
#             — flags shared keys and repeated aliases;
#               city is used only to build candidate pairs
#    Layer 2: Levenshtein runs ONLY on candidate pairs from same
#             access key or same city block (not all pairs)
#
#    Complexity: O(N + Σ block_size² × L²)
#    where N = records, L = alias length, block = same-key or same-city group
# ─────────────────────────────────────────────────────────────────────
def hash_levenshtein(agents, threshold=2):
    t0 = time.perf_counter()

    by_key   = defaultdict(list)
    by_city  = defaultdict(list)
    by_alias = defaultdict(list)

    for agent in agents:
        by_key[agent["Access_Key"]].append(agent)
        by_city[agent["Last_Known_City"]].append(agent)
        by_alias[agent["Alias"].lower()].append(agent)

    suspicious     = defaultdict(list)
    near_dup_flags = []
    exact_dup_flags = []

    # --- Layer 1a: Shared access key ---
    for key, records in by_key.items():
        if len(records) > 1:
            for r in records:
                suspicious[r["Agent_ID"]].append(f"Shared access key: {key}")

    # --- Layer 1b: Repeated alias (exact) ---
    for alias, records in by_alias.items():
        if len(records) > 1:
            for r in records:
                suspicious[r["Agent_ID"]].append(f"Repeated alias: {r['Alias']}")
            for a, b in combinations(records, 2):
                exact_dup_flags.append((a, b, 0))

    # --- Layer 2: Build candidate pairs from same key OR same city only ---
    # Note: city is used for candidate pair building only, not direct flagging
    candidate_pairs = set()
    id_to_agent = {a["Agent_ID"]: a for a in agents}

    def add_pairs(records):
        for a, b in combinations(records, 2):
            pair = tuple(sorted([a["Agent_ID"], b["Agent_ID"]]))
            candidate_pairs.add(pair)

    for records in by_key.values():
        if len(records) > 1:
            add_pairs(records)

    for records in by_city.values():
        if len(records) > 1:
            add_pairs(records)

    # --- Layer 2: Levenshtein on candidate pairs only ---
    for id1, id2 in candidate_pairs:
        a = id_to_agent[id1]
        b = id_to_agent[id2]
        al, bl = a["Alias"].lower(), b["Alias"].lower()

        if abs(len(al) - len(bl)) > threshold:
            continue

        dist = levenshtein(al, bl)
        is_prefix_related = al.startswith(bl) or bl.startswith(al)

        if dist == 0:
            continue
        if dist <= 1 or (dist <= threshold and is_prefix_related):
            near_dup_flags.append((a, b, dist))
            suspicious[a["Agent_ID"]].append(
                f"Near-duplicate alias with {b['Agent_ID']} (distance {dist})")
            suspicious[b["Agent_ID"]].append(
                f"Near-duplicate alias with {a['Agent_ID']} (distance {dist})")

    # Build flagged list with reasons
    flagged = []
    for agent in agents:
        if agent["Agent_ID"] in suspicious:
            record = agent.copy()
            record["Reason"] = "; ".join(sorted(set(suspicious[agent["Agent_ID"]])))
            flagged.append(record)

    elapsed = time.perf_counter() - t0
    key_clusters = sorted(by_key.items(), key=lambda x: len(x[1]), reverse=True)
    return flagged, exact_dup_flags, near_dup_flags, key_clusters, elapsed

## Step 5 — Algorithm 2: Naive String Matching + Rabin-Karp (Merged) (Rejected)

In [5]:
# ─────────────────────────────────────────────────────────────────────
# 4. ALGORITHM 2 — Naive String Matching + Rabin-Karp (REJECTED)
#    Step 1: Rabin-Karp rolling hash detects exact alias duplicates
#            and shared access keys using h = (h * BASE + ord(c)) % MOD.
#    Step 2: Naive String Matching runs on remaining unmatched pairs —
#            compares aliases character by character for exact sequence
#            matches — O(N² × L).
#    Limitation: Rabin-Karp misses near-duplicates because similar strings
#    hash to completely different values. Naive Matching only performs exact
#    sequence comparison — neither step can detect aliases like Falcon vs
#    Falcon_1. The merged approach still fails the near-duplicate use case.
# ─────────────────────────────────────────────────────────────────────
def naive_rabin_karp(agents):
    t0 = time.perf_counter()
    BASE, MOD = 31, 10**9 + 9

    def rk_hash(s):
        h = 0
        for c in s.lower():
            h = (h * BASE + ord(c)) % MOD
        return h

    # Step 1: Rabin-Karp — detect exact alias duplicates via rolling hash
    #         and flag shared access keys
    hash_map = defaultdict(list)
    key_map  = defaultdict(list)
    for a in agents:
        hash_map[rk_hash(a["Alias"])].append(a)
        key_map[a["Access_Key"]].append(a)

    suspicious  = set()
    matched_ids = set()
    for records in hash_map.values():
        if len({r["Agent_ID"] for r in records}) > 1:
            for r in records:
                suspicious.add(r["Agent_ID"])
                matched_ids.add(r["Agent_ID"])
    for records in key_map.values():
        if len({r["Agent_ID"] for r in records}) > 1:
            for r in records:
                suspicious.add(r["Agent_ID"])
                matched_ids.add(r["Agent_ID"])

    # Step 2: Naive String Matching — compare remaining unmatched pairs
    #         character by character for exact sequence matches
    unmatched = [a for a in agents if a["Agent_ID"] not in matched_ids]
    for i in range(len(unmatched)):
        for j in range(i + 1, len(unmatched)):
            a1, a2 = unmatched[i]["Alias"].lower(), unmatched[j]["Alias"].lower()
            if len(a1) != len(a2):
                continue
            if all(a1[k] == a2[k] for k in range(len(a1))):
                suspicious.add(unmatched[i]["Agent_ID"])
                suspicious.add(unmatched[j]["Agent_ID"])

    elapsed = time.perf_counter() - t0
    return [a for a in agents if a["Agent_ID"] in suspicious], elapsed

## Step 6 — Algorithm 3: Trie + Linear Search (Merged) (Rejected)

In [6]:
# ─────────────────────────────────────────────────────────────────────
# 5. ALGORITHM 3 — Trie + Linear Search (REJECTED)
#    Step 1: Trie inserts all aliases and detects exact duplicates at
#            leaf nodes via DFS — only catches identical alias strings.
#    Step 2: Linear Search scans every pair of records for shared access
#            keys using a nested loop — O(N²), adds unnecessary cost.
#    Limitation: Trie only catches leaf-node exact duplicates and misses
#    aliases that differ in the middle or end (e.g. Falcon vs Falcon_1).
#    Linear Search adds O(N²) cost for key comparison that a hash table
#    handles in O(N) — the merge compounds cost without improving recall.
# ─────────────────────────────────────────────────────────────────────
class TrieNode:
    def __init__(self):
        self.children = {}
        self.records  = []

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, alias, record):
        node = self.root
        for char in alias.lower():
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.records.append(record)

    def collect_exact_duplicates(self):
        suspicious = set()
        def dfs(node):
            if len({r["Agent_ID"] for r in node.records}) > 1:
                for r in node.records: suspicious.add(r["Agent_ID"])
            for child in node.children.values(): dfs(child)
        dfs(self.root)
        return suspicious

def trie_linear(agents):
    t0 = time.perf_counter()

    # Step 1: Trie — insert all aliases and detect exact duplicates at leaf nodes via DFS
    trie = Trie()
    for a in agents: trie.insert(a["Alias"], a)
    suspicious = trie.collect_exact_duplicates()

    # Step 2: Linear Search — scan every pair of records for shared access keys (O(N²))
    for i in range(len(agents)):
        for j in range(i + 1, len(agents)):
            if agents[i]["Access_Key"] == agents[j]["Access_Key"]:
                suspicious.add(agents[i]["Agent_ID"])
                suspicious.add(agents[j]["Agent_ID"])

    elapsed = time.perf_counter() - t0
    return [a for a in agents if a["Agent_ID"] in suspicious], elapsed

## Step 7 — Run All Three Algorithms

In [7]:
# ─────────────────────────────────────────────────────────────────────
# 6. Run All Three Algorithms
# ─────────────────────────────────────────────────────────────────────
print("Running all three algorithms...\n")
hl_flagged, hl_exact, hl_near, hl_clusters, hl_time = hash_levenshtein(agents, threshold=2)
rk_flagged, rk_time = naive_rabin_karp(agents)
tr_flagged, tr_time = trie_linear(agents)

hl_ids = {r["Agent_ID"] for r in hl_flagged}
rk_ids = {r["Agent_ID"] for r in rk_flagged}
tr_ids = {r["Agent_ID"] for r in tr_flagged}

rk_extra = [a for a in agents if a["Agent_ID"] in (rk_ids - hl_ids)]
tr_extra  = [a for a in agents if a["Agent_ID"] in (tr_ids - hl_ids)]

Running all three algorithms...



## Step 8 — Results & Analysis

In [8]:
# ─────────────────────────────────────────────────────────────────────
# 7. Ranked Cluster Table
# ─────────────────────────────────────────────────────────────────────
print("=" * 72)
print("  RANKED SUSPICIOUS CLUSTERS  (Hash Table — Access Key frequency)")
print("=" * 72)
cluster_rows = []
for rank, (key, records) in enumerate(hl_clusters, 1):
    ids  = ", ".join(r["Agent_ID"] for r in records)
    flag = "YES (!)" if len({r["Agent_ID"] for r in records}) > 1 else "No"
    cluster_rows.append([rank, key, len(records), ids, flag])
print(tabulate(cluster_rows,
               headers=["Rank", "Access Key", "Count", "Agent IDs", "Suspicious?"],
               tablefmt="rounded_outline"))

  RANKED SUSPICIOUS CLUSTERS  (Hash Table — Access Key frequency)
╭────────┬──────────────┬─────────┬─────────────────────┬───────────────╮
│   Rank │ Access Key   │   Count │ Agent IDs           │ Suspicious?   │
├────────┼──────────────┼─────────┼─────────────────────┼───────────────┤
│      1 │ K9X4         │       3 │ A1023, A3102, A9450 │ YES (!)       │
│      2 │ M2P7         │       2 │ A2088, A9012        │ YES (!)       │
│      3 │ T7Q1         │       2 │ A1194, A2876        │ YES (!)       │
│      4 │ Z1L9         │       2 │ A5127, A8044        │ YES (!)       │
│      5 │ P5H6         │       2 │ A3884, A9771        │ YES (!)       │
│      6 │ B8D3         │       1 │ A4410               │ No            │
│      7 │ B8D8         │       1 │ A6401               │ No            │
│      8 │ R4C2         │       1 │ A7230               │ No            │
╰────────┴──────────────┴─────────┴─────────────────────┴───────────────╯


In [9]:
# ─────────────────────────────────────────────────────────────────────
# 8. Near-Duplicate Aliases
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  NEAR-DUPLICATE ALIASES  (Levenshtein — threshold <= 2, candidate pairs only)")
print("=" * 72)
if hl_near:
    near_rows = [[a["Agent_ID"], a["Alias"], b["Agent_ID"], b["Alias"], dist]
                 for a, b, dist in hl_near]
    print(tabulate(near_rows,
                   headers=["Agent ID 1", "Alias 1", "Agent ID 2", "Alias 2", "Distance"],
                   tablefmt="rounded_outline"))
else:
    print("  No near-duplicate aliases detected.")


  NEAR-DUPLICATE ALIASES  (Levenshtein — threshold <= 2, candidate pairs only)
╭──────────────┬───────────┬──────────────┬───────────┬────────────╮
│ Agent ID 1   │ Alias 1   │ Agent ID 2   │ Alias 2   │   Distance │
├──────────────┼───────────┼──────────────┼───────────┼────────────┤
│ A1194        │ Falcon    │ A2876        │ Falcon_1  │          2 │
╰──────────────┴───────────┴──────────────┴───────────┴────────────╯


In [10]:
# ─────────────────────────────────────────────────────────────────────
# 9. Flagged Records per Algorithm
# ─────────────────────────────────────────────────────────────────────
HEADERS    = ["Agent ID", "Alias", "Access Key", "City", "Status", "Linked Site"]
HEADERS_HL = ["Agent ID", "Alias", "Access Key", "City", "Status", "Linked Site", "Reason"]

def make_rows(records, include_reason=False):
    rows = []
    for r in records:
        row = [r["Agent_ID"], r["Alias"], r["Access_Key"],
               r["Last_Known_City"], r["Status"], r["Linked_Site"]]
        if include_reason:
            row.append(r.get("Reason", ""))
        rows.append(row)
    return rows

print("\n" + "=" * 72)
print("  HASH TABLE + LEVENSHTEIN — Suspicious Records  <- CHOSEN")
print("=" * 72)
print(tabulate(make_rows(hl_flagged, include_reason=True),
               headers=HEADERS_HL, tablefmt="rounded_outline"))

print("\n" + "=" * 72)
print("  NAIVE STRING MATCHING + RABIN-KARP — Suspicious Records")
print("=" * 72)
print(tabulate(make_rows(rk_flagged), headers=HEADERS, tablefmt="rounded_outline"))

print("\n" + "=" * 72)
print("  TRIE + LINEAR SEARCH — Suspicious Records")
print("=" * 72)
print(tabulate(make_rows(tr_flagged), headers=HEADERS, tablefmt="rounded_outline"))


  HASH TABLE + LEVENSHTEIN — Suspicious Records  <- CHOSEN
╭────────────┬──────────┬──────────────┬────────────┬────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────╮
│ Agent ID   │ Alias    │ Access Key   │ City       │ Status     │ Linked Site           │ Reason                                                                │
├────────────┼──────────┼──────────────┼────────────┼────────────┼───────────────────────┼───────────────────────────────────────────────────────────────────────┤
│ A1023      │ Raven    │ K9X4         │ Blackridge │ Active     │ East Daran Depot      │ Repeated alias: Raven; Shared access key: K9X4                        │
│ A2088      │ Viper    │ M2P7         │ East Daran │ Active     │ Greyfen Reach         │ Repeated alias: Viper; Shared access key: M2P7                        │
│ A1194      │ Falcon   │ T7Q1         │ Norvale    │ Missing    │ North Cargo Pier      │ Near-duplicate alias with A2876 (d

In [11]:
# ─────────────────────────────────────────────────────────────────────
# 10. Algorithm Comparison
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  ALGORITHM COMPARISON")
print("=" * 72)
# Count near-duplicate pairs per algorithm (only chosen detects them)
hl_near_count = len(hl_near)
rk_near_count = 0   # Naive String Matching + Rabin-Karp cannot detect near-duplicates
tr_near_count = 0   # Trie + Linear Search cannot detect mid/end differences

comp_rows = [
    ["Hash Table + Levenshtein <- CHOSEN",     "O(N + C x L²)", f"{hl_time*1000:.4f}", len(hl_flagged), hl_near_count, 0],
    ["Naive String Matching + Rabin-Karp",      "O(N² x L)",            f"{rk_time*1000:.4f}", len(rk_flagged), rk_near_count, len(rk_extra)],
    ["Trie + Linear Search",                    "O(N²)", f"{tr_time*1000:.4f}", len(tr_flagged), tr_near_count, len(tr_extra)],
]
print(tabulate(comp_rows,
               headers=["Algorithm", "Complexity", "Time (ms)", "Flagged", "Near-dup Pairs", "Extra Flagged"],
               tablefmt="rounded_outline"))


  ALGORITHM COMPARISON
╭────────────────────────────────────┬───────────────┬─────────────┬───────────┬──────────────────┬─────────────────╮
│ Algorithm                          │ Complexity    │   Time (ms) │   Flagged │   Near-dup Pairs │   Extra Flagged │
├────────────────────────────────────┼───────────────┼─────────────┼───────────┼──────────────────┼─────────────────┤
│ Hash Table + Levenshtein <- CHOSEN │ O(N + C x L²) │      0.1338 │        13 │                1 │               0 │
│ Naive String Matching + Rabin-Karp │ O(N² x L)     │      0.0309 │        13 │                0 │               0 │
│ Trie + Linear Search               │ O(N²)         │      0.0498 │        13 │                0 │               0 │
╰────────────────────────────────────┴───────────────┴─────────────┴───────────┴──────────────────┴─────────────────╯


In [12]:
# ─────────────────────────────────────────────────────────────────────
# 11. Extra Records Flagged by Rejected Algorithms
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  EXTRA RECORDS FLAGGED BY REJECTED ALGORITHMS")
print("  (Records flagged by rejected algorithms but not by chosen)")
print("=" * 72)

print("\n  Naive String Matching + Rabin-Karp extra:")
if rk_extra:
    print(tabulate(make_rows(rk_extra), headers=HEADERS, tablefmt="rounded_outline"))
    print("  Note: Rabin-Karp flags agents with same alias hash regardless of access key.")
    print("        Naive Matching only catches exact same-length sequences.")
    print("        Neither step detects near-duplicate aliases like Falcon vs Falcon_1.")
else:
    print("  None")

print("\n  Trie + Linear Search extra:")
if tr_extra:
    print(tabulate(make_rows(tr_extra), headers=HEADERS, tablefmt="rounded_outline"))
    print("  Note: Trie flags agents sharing an alias prefix without verifying")
    print("        access key or city consistency.")
else:
    print("  None")


  EXTRA RECORDS FLAGGED BY REJECTED ALGORITHMS
  (Records flagged by rejected algorithms but not by chosen)

  Naive String Matching + Rabin-Karp extra:
  None

  Trie + Linear Search extra:
  None


In [13]:
# ─────────────────────────────────────────────────────────────────────
# 12. Rejection Analysis
# ─────────────────────────────────────────────────────────────────────
print(f"""
{'='*72}
  REJECTION ANALYSIS  (n = {len(agents)} records, Levenshtein threshold = 2)
{'='*72}
  Chosen — Hash Table + Levenshtein Distance:
    Layer 1 (Hash Table) groups by access key, city, and alias. It flags
    shared keys and repeated aliases, while city is used as a blocking key
    for candidate comparison only — not as direct suspicious evidence.
    Layer 2 (Levenshtein) runs only on candidate pairs within same access key
    or same city block — not all pairs. Complexity: O(N + Σ block² x L²).
    Threshold set to 2 to cover single-character edits and suffix patterns
    like Falcon vs Falcon_1. Each record is indexed once using hash tables;
    Levenshtein verification is then applied only to candidate pairs.

  Rejected — Naive String Matching + Rabin-Karp ({rk_time/hl_time:.1f}x relative to chosen):
    Step 1 (Rabin-Karp) uses rolling hash to detect exact alias duplicates
    and shared access keys, but misses near-duplicates entirely — similar
    strings like Falcon and Falcon_1 produce completely different hash values.
    Step 2 (Naive Matching) compares remaining unmatched pairs character by
    character, but only detects exact same-length sequences — it cannot
    catch aliases that differ by an insertion, deletion, or suffix. The
    merged approach is O(N² x L) in the worst case and still fails to
    identify the Falcon vs Falcon_1 near-duplicate pair.
    The merged approach may still flag Falcon and Falcon_1 through their 
    shared access key T7Q1, but it does not recognise them as a 
    near-duplicate alias pair because no edit-distance comparison is 
    performed.

  Rejected — Trie + Linear Search ({tr_time/hl_time:.1f}x relative to chosen):
    Step 1 (Trie) inserts all aliases and detects exact duplicates at leaf
    nodes via DFS, but misses any aliases that differ in the middle or end
    of the string — Falcon vs Falcon_1 is not caught since they diverge
    at the suffix and land on different leaf nodes.
    The merged approach may still flag Falcon and Falcon_1 through their 
    shared access key T7Q1, but it does not recognise them as a 
    near-duplicate alias pair because no edit-distance comparison is 
    performed.
    Step 2 (Linear Search) scans every pair of records for shared access
    keys in a nested O(N²) loop, adding unnecessary quadratic cost for a
    check that a hash table completes in O(N). The merge compounds cost
    without improving recall. Near-duplicate pairs found: 0.
""")


  REJECTION ANALYSIS  (n = 14 records, Levenshtein threshold = 2)
  Chosen — Hash Table + Levenshtein Distance:
    Layer 1 (Hash Table) groups by access key, city, and alias. It flags
    shared keys and repeated aliases, while city is used as a blocking key
    for candidate comparison only — not as direct suspicious evidence.
    Layer 2 (Levenshtein) runs only on candidate pairs within same access key
    or same city block — not all pairs. Complexity: O(N + Σ block² x L²).
    Threshold set to 2 to cover single-character edits and suffix patterns
    like Falcon vs Falcon_1. Each record is indexed once using hash tables;
    Levenshtein verification is then applied only to candidate pairs.

  Rejected — Naive String Matching + Rabin-Karp (0.2x relative to chosen):
    Step 1 (Rabin-Karp) uses rolling hash to detect exact alias duplicates
    and shared access keys, but misses near-duplicates entirely — similar
    strings like Falcon and Falcon_1 produce completely different hash

In [14]:
print("\n" + "=" * 72)
print("  PART 2 — FINAL SUMMARY")
print("=" * 72)
summary_rows = [
    ["Total records loaded (Sheet B)",          len(agents)],
    ["Chosen algorithm",                        "Hash Table + Levenshtein"],
    ["Candidate pairs checked",                 8],
    ["Brute-force pairs (avoided)",             91],
    ["Near-duplicate pairs found (chosen)",     len(hl_near)],
    ["Near-duplicate pairs found (rejected)",   0],
    ["Flagged records (chosen)",                len(hl_flagged)],
]
print(tabulate(summary_rows, headers=["Metric", "Result"], tablefmt="rounded_outline"))


  PART 2 — FINAL SUMMARY
╭───────────────────────────────────────┬──────────────────────────╮
│ Metric                                │ Result                   │
├───────────────────────────────────────┼──────────────────────────┤
│ Total records loaded (Sheet B)        │ 14                       │
│ Chosen algorithm                      │ Hash Table + Levenshtein │
│ Candidate pairs checked               │ 8                        │
│ Brute-force pairs (avoided)           │ 91                       │
│ Near-duplicate pairs found (chosen)   │ 1                        │
│ Near-duplicate pairs found (rejected) │ 0                        │
│ Flagged records (chosen)              │ 13                       │
╰───────────────────────────────────────┴──────────────────────────╯


# Part 7 — The Phantom Dice

## Step 1 — Mission Problem and Candidate Algorithms

**Resource:** Infiltration sector — one sector the team will physically enter.

**Constraint:** Cypher Nexus has flagged certain sectors as monitored. If the team repeatedly uses a predictable entry point, the enemy can learn the pattern and intercept them.

**Goal:** Select a real movement sector that is safe, deception-aware, and non-deterministic, while also selecting a decoy sector to misdirect surveillance.

### Candidate Algorithms

| Status | Algorithm | Main Idea |
|---|---|---|
| CHOSEN | Dual-Objective Sector Scoring with Weighted Prefix Draw | Combines danger score, safety weight, deception bonus, prediction penalty, and prefix-sum random draw. |
| REJECTED | Two-Phase Elimination + Weighted Draw | Filters high-danger predicted sectors first, then performs weighted random selection on survivors. |
| REJECTED | Danger-Inverted Prefix Sum | Uses inverse danger and prediction penalty, but ignores Decoy_Value. |

## Step 2 — Dataset Loading and Preview

In [15]:
# ---------------------------------------------------------------------
# 0. Install & Imports
# ---------------------------------------------------------------------
!pip install openpyxl tabulate --quiet

import os
import random
import time
import openpyxl
import pandas as pd
from tabulate import tabulate


In [16]:
# ─────────────────────────────────────────────────────────────────────
# 1. Dataset Loading and Preview
# ─────────────────────────────────────────────────────────────────────
EXCEL_PATH = None
search_roots = ["/content/Algo-Project", os.getcwd()]
for search_root in search_roots:
    if not os.path.exists(search_root):
        continue
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if "7" in file and file.endswith(".xlsx") and not file.startswith("."):
                EXCEL_PATH = os.path.join(root, file)
                break
        if EXCEL_PATH is not None:
            break
    if EXCEL_PATH is not None:
        break

if EXCEL_PATH is None:
    raise FileNotFoundError("Part 7 dataset not found. Check your repo structure.")
print(f"Dataset found: {EXCEL_PATH}")

SHEET_NAME = "B"
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

sector_columns = [
    "Sector",
    "Patrol_Frequency",
    "Thermal_Scan_Level",
    "Drone_Coverage",
    "Predicted_By_Enemy",
    "Decoy_Value",
]

sectors = []
for row in ws.iter_rows(min_row=3, values_only=True):
    if row[1] is None or row[1] == "Sector":
        continue
    sectors.append({
        "Sector"            : row[1],
        "Patrol_Frequency"  : row[2],
        "Thermal_Scan_Level": row[3],
        "Drone_Coverage"    : row[4],
        "Predicted_By_Enemy": row[5],
        "Decoy_Value"       : row[6],
    })

sector_df = pd.DataFrame(sectors, columns=sector_columns)
print(f"Loaded {len(sector_df)} sector records from Sheet '{SHEET_NAME}'\n")
print("=" * 95)
print("  PART 7 — THE PHANTOM DICE: DATASET PREVIEW")
print("=" * 95)
print(tabulate(sector_df[sector_columns], headers="keys", tablefmt="rounded_outline", showindex=False))

Dataset found: /content/Algo-Project/Datasets/Part 7.xlsx
Loaded 8 sector records from Sheet 'B'

  PART 7 — THE PHANTOM DICE: DATASET PREVIEW
╭──────────┬────────────────────┬──────────────────────┬──────────────────┬──────────────────────┬───────────────╮
│ Sector   │   Patrol_Frequency │   Thermal_Scan_Level │   Drone_Coverage │ Predicted_By_Enemy   │   Decoy_Value │
├──────────┼────────────────────┼──────────────────────┼──────────────────┼──────────────────────┼───────────────┤
│ S1       │                  2 │                    2 │                2 │ Yes                  │             4 │
│ S2       │                  5 │                    4 │                5 │ No                   │             8 │
│ S3       │                  7 │                    6 │                8 │ Yes                  │             3 │
│ S4       │                  4 │                    3 │                4 │ No                   │             7 │
│ S5       │                  6 │                   

## Step 3 — Define Helper Functions

In [17]:
def _as_sector_frame(sector_data):
    if isinstance(sector_data, pd.DataFrame):
        frame = sector_data.copy()
    else:
        frame = pd.DataFrame(sector_data)
    required = [
        "Sector", "Patrol_Frequency", "Thermal_Scan_Level",
        "Drone_Coverage", "Predicted_By_Enemy", "Decoy_Value"
    ]
    missing = [col for col in required if col not in frame.columns]
    if missing:
        raise ValueError(f"Missing required Part 7 columns: {missing}")
    for col in ["Patrol_Frequency", "Thermal_Scan_Level", "Drone_Coverage", "Decoy_Value"]:
        frame[col] = pd.to_numeric(frame[col])
    return frame[required].reset_index(drop=True)

def _draw_from_prefix(prefix_values):
    r = random.random()
    selected_idx = len(prefix_values) - 1
    for idx, threshold in enumerate(prefix_values):
        if r <= threshold:
            selected_idx = idx
            break
    return r, selected_idx

## Step 4 — Algorithm 1: Dual-Objective Sector Scoring with Weighted Prefix Draw (Chosen)

In [18]:
def dual_objective_prefix_draw(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)

    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    max_decoy = table["Decoy_Value"].max()
    table["Safety_Weight"] = 1 / table["Danger_Score"]
    table["Deception_Bonus"] = 1 + (table["Decoy_Value" ] / max_decoy if max_decoy else 0)
    table["Composite_Score"] = table["Safety_Weight"] * table["Deception_Bonus"]
    table.loc[table["Predicted_By_Enemy"] == "Yes", "Composite_Score"] /= 2
    table["Selection_Probability"] = table["Composite_Score"] / table["Composite_Score"].sum()
    table["Prefix_Probability"] = table["Selection_Probability"].cumsum()

    r, selected_idx = _draw_from_prefix(table["Prefix_Probability"])
    selected_real_sector = table.iloc[selected_idx].to_dict()
    real_sector_name = selected_real_sector["Sector"]

    predicted_candidates = table[
        (table["Predicted_By_Enemy"] == "Yes")
        & (table["Sector"] != real_sector_name)
    ]
    if predicted_candidates.empty:
        predicted_candidates = table[table["Sector"] != real_sector_name]
    decoy_sector = predicted_candidates.sort_values(
        ["Decoy_Value", "Composite_Score"], ascending=[False, False]
    ).iloc[0].to_dict()

    elapsed = time.perf_counter() - t0
    return {
        "selected_real_sector": selected_real_sector,
        "decoy_sector": decoy_sector,
        "probability_table": table,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 5 — Algorithm 2: Two-Phase Elimination + Weighted Draw (Rejected)

In [19]:
def two_phase_elimination_weighted_draw(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)
    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    sorted_dangers = sorted(table["Danger_Score"].tolist())
    middle = len(sorted_dangers) // 2
    if len(sorted_dangers) % 2 == 0:
        median_danger = (sorted_dangers[middle - 1] + sorted_dangers[middle]) / 2
    else:
        median_danger = sorted_dangers[middle]
    table["Eliminated"] = (
        (table["Danger_Score"] > median_danger)
        & (table["Predicted_By_Enemy"] == "Yes")
    )
    table["Phase_Result"] = table["Eliminated"].map({True: "Eliminated", False: "Survived"})

    survivors = table[~table["Eliminated"]].copy()
    if survivors.empty:
        survivors = table.copy()
        survivors["Phase_Result"] = "Fallback survivor"

    survivors["Weight"] = 1 / survivors["Danger_Score"]
    survivors["Selection_Probability"] = survivors["Weight"] / survivors["Weight"].sum()
    survivors["Prefix_Probability"] = survivors["Selection_Probability"].cumsum()

    r, selected_position = _draw_from_prefix(survivors["Prefix_Probability"])
    selected_sector = survivors.iloc[selected_position].to_dict()
    elapsed = time.perf_counter() - t0
    return {
        "selected_sector": selected_sector,
        "median_danger": median_danger,
        "elimination_table": table,
        "survivor_table": survivors,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 6 — Algorithm 3: Danger-Inverted Prefix Sum (Rejected)

In [20]:
def danger_inverted_prefix_sum(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)
    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    table["Prediction_Penalty"] = table["Predicted_By_Enemy"].map({"Yes": 0.5, "No": 1.0}).fillna(1.0)
    table["Weight"] = (1 / table["Danger_Score"]) * table["Prediction_Penalty"]
    table["Selection_Probability"] = table["Weight"] / table["Weight"].sum()
    table["Prefix_Probability"] = table["Selection_Probability"].cumsum()

    r, selected_idx = _draw_from_prefix(table["Prefix_Probability"])
    selected_sector = table.iloc[selected_idx].to_dict()
    elapsed = time.perf_counter() - t0
    return {
        "selected_sector": selected_sector,
        "probability_table": table,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 7 — Run All Candidate Algorithms

In [21]:
# ─────────────────────────────────────────────────────────────────────
# 3. Run All Three Candidate Algorithms
# ─────────────────────────────────────────────────────────────────────
print("Running all three Part 7 algorithms...\n")
dual_result = dual_objective_prefix_draw(sector_df)
two_phase_result = two_phase_elimination_weighted_draw(sector_df)
danger_inverted_result = danger_inverted_prefix_sum(sector_df)

real_sector = dual_result["selected_real_sector"]
decoy_sector = dual_result["decoy_sector"]
two_phase_sector = two_phase_result["selected_sector"]
danger_inverted_sector = danger_inverted_result["selected_sector"]

Running all three Part 7 algorithms...



## Step 8 — Chosen Algorithm Output and Probability Table

In [22]:
# ─────────────────────────────────────────────────────────────────────
# 4. Chosen Algorithm Output: Dual-Objective Sector Scoring
# ─────────────────────────────────────────────────────────────────────
chosen_table = dual_result["probability_table"]
chosen_display_columns = [
    "Sector",
    "Patrol_Frequency",
    "Thermal_Scan_Level",
    "Drone_Coverage",
    "Danger_Score",
    "Predicted_By_Enemy",
    "Decoy_Value",
    "Safety_Weight",
    "Deception_Bonus",
    "Composite_Score",
    "Selection_Probability",
    "Prefix_Probability",
]

print("=" * 120)
print("  CHOSEN ALGORITHM: DUAL-OBJECTIVE SECTOR SCORING WITH WEIGHTED PREFIX DRAW")
print("=" * 120)
print("danger_score = Patrol_Frequency + Thermal_Scan_Level + Drone_Coverage")
print("safety_weight = 1 / danger_score")
print("deception_bonus = 1 + (Decoy_Value / max_Decoy_Value)")
print("score = safety_weight × deception_bonus")
print("if Predicted_By_Enemy == Yes: score = score / 2")
print("probability = score / total_score\n")
print(tabulate(
    chosen_table[chosen_display_columns],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))
print("\nNote: Prefix_Probability of last sector = 1.0000 confirms all probabilities sum correctly to 1.")

  CHOSEN ALGORITHM: DUAL-OBJECTIVE SECTOR SCORING WITH WEIGHTED PREFIX DRAW
danger_score = Patrol_Frequency + Thermal_Scan_Level + Drone_Coverage
safety_weight = 1 / danger_score
deception_bonus = 1 + (Decoy_Value / max_Decoy_Value)
score = safety_weight × deception_bonus
if Predicted_By_Enemy == Yes: score = score / 2
probability = score / total_score

╭──────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────┬──────────────────────┬───────────────┬─────────────────┬───────────────────┬───────────────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Patrol_Frequency │   Thermal_Scan_Level │   Drone_Coverage │   Danger_Score │ Predicted_By_Enemy   │   Decoy_Value │   Safety_Weight │   Deception_Bonus │   Composite_Score │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────────┼──────────────────────┼──────────────────┼────────────────┼──────────────────────┼───────────────┼─────────────────┼─────────────────

In [23]:
# ---------------------------------------------------------------------
# 5. Chosen Algorithm Result: Real Sector and Decoy Sector
# ---------------------------------------------------------------------
print("=" * 95)
print("  SELECTED REAL SECTOR & DECOY SECTOR")
print("=" * 95)
print(f"Random value r                 : {dual_result['random_value']:.6f}")
print(f"Selected real sector           : {real_sector['Sector']}")
print(f"Real sector danger score       : {real_sector['Danger_Score']}")
print(f"Real sector Decoy_Value        : {real_sector['Decoy_Value']}")
print(f"Real sector predicted status   : {real_sector['Predicted_By_Enemy']}")
print(f"Decoy sector                   : {decoy_sector['Sector']}")
print(f"Decoy sector danger score      : {decoy_sector['Danger_Score']}")
print(f"Decoy sector Decoy_Value       : {decoy_sector['Decoy_Value']}")
print(f"Decoy sector predicted status  : {decoy_sector['Predicted_By_Enemy']}")
print(f"Time taken                     : {dual_result['elapsed'] * 1000:.4f} ms")
print("Note: Because this algorithm is non-deterministic, the selected real sector may differ each run.")


  SELECTED REAL SECTOR & DECOY SECTOR
Random value r                 : 0.331647
Selected real sector           : S4
Real sector danger score       : 11
Real sector Decoy_Value        : 7
Real sector predicted status   : No
Decoy sector                   : S1
Decoy sector danger score      : 6
Decoy sector Decoy_Value       : 4
Decoy sector predicted status  : Yes
Time taken                     : 19.1154 ms
Note: Because this algorithm is non-deterministic, the selected real sector may differ each run.


## Step 9 — Rejected Algorithm Outputs

### Step 9.1 — Two-Phase Elimination Output

In [24]:
# ─────────────────────────────────────────────────────────────────────
# 5. Rejected Algorithm Output: Two-Phase Elimination + Weighted Draw
# ─────────────────────────────────────────────────────────────────────
elimination_table = two_phase_result["elimination_table"]
survivor_table = two_phase_result["survivor_table"]
eliminated = elimination_table[elimination_table["Eliminated"]]

print("=" * 105)
print("  REJECTED ALGORITHM: TWO-PHASE ELIMINATION + WEIGHTED DRAW")
print("=" * 105)
print(f"Median danger score            : {two_phase_result['median_danger']:.4f}")
print(f"Random value r                 : {two_phase_result['random_value']:.6f}")
print(f"Selected sector from survivors : {two_phase_sector['Sector']}")
print(f"Time taken                     : {two_phase_result['elapsed'] * 1000:.4f} ms\n")

print("Survived sectors:")
print(tabulate(
    survivor_table[["Sector", "Danger_Score", "Predicted_By_Enemy", "Weight", "Selection_Probability", "Prefix_Probability"]],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))

print("\nEliminated sectors:")
if eliminated.empty:
    print("No sectors eliminated.")
else:
    print(tabulate(
        eliminated[["Sector", "Danger_Score", "Predicted_By_Enemy", "Decoy_Value", "Phase_Result"]],
        headers="keys",
        tablefmt="rounded_outline",
        showindex=False,
        floatfmt=".4f",
    ))

  REJECTED ALGORITHM: TWO-PHASE ELIMINATION + WEIGHTED DRAW
Median danger score            : 14.0000
Random value r                 : 0.386321
Selected sector from survivors : S2
Time taken                     : 11.4410 ms

Survived sectors:
╭──────────┬────────────────┬──────────────────────┬──────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Danger_Score │ Predicted_By_Enemy   │   Weight │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────┼──────────────────────┼──────────┼─────────────────────────┼──────────────────────┤
│ S1       │              6 │ Yes                  │   0.1667 │                  0.3029 │               0.3029 │
│ S2       │             14 │ No                   │   0.0714 │                  0.1298 │               0.4328 │
│ S4       │             11 │ No                   │   0.0909 │                  0.1652 │               0.5980 │
│ S5       │             17 │ No                   │   0.0588 │                 

### Step 9.2 — Danger-Inverted Prefix Sum Output

In [25]:
# ─────────────────────────────────────────────────────────────────────
# 6. Rejected Algorithm Output: Danger-Inverted Prefix Sum
# ─────────────────────────────────────────────────────────────────────
danger_table = danger_inverted_result["probability_table"]
print("=" * 105)
print("  REJECTED ALGORITHM: DANGER-INVERTED PREFIX SUM")
print("=" * 105)
print(f"Random value r                 : {danger_inverted_result['random_value']:.6f}")
print(f"Selected sector                : {danger_inverted_sector['Sector']}")
print(f"Time taken                     : {danger_inverted_result['elapsed'] * 1000:.4f} ms\n")
print(tabulate(
    danger_table[["Sector", "Danger_Score", "Weight", "Prediction_Penalty", "Selection_Probability", "Prefix_Probability"]],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))

  REJECTED ALGORITHM: DANGER-INVERTED PREFIX SUM
Random value r                 : 0.962622
Selected sector                : S8
Time taken                     : 4.1859 ms

╭──────────┬────────────────┬──────────┬──────────────────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Danger_Score │   Weight │   Prediction_Penalty │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────┼──────────┼──────────────────────┼─────────────────────────┼──────────────────────┤
│ S1       │              6 │   0.0833 │               0.5000 │                  0.1623 │               0.1623 │
│ S2       │             14 │   0.0714 │               1.0000 │                  0.1391 │               0.3015 │
│ S3       │             21 │   0.0238 │               0.5000 │                  0.0464 │               0.3478 │
│ S4       │             11 │   0.0909 │               1.0000 │                  0.1771 │               0.5249 │
│ S5       │             17 │   0.0588

## Step 10 — Time and Space Complexity Analysis

**Dual-Objective Sector Scoring:**

`T_total = O(n) + O(n) + O(n) + O(n) + O(n) + O(n) = O(n)`

`Space = O(n)`

**Two-Phase Elimination + Weighted Draw:**

`T_total = O(n) + O(n log n) + O(n) + O(n) = O(n log n)`

`Space = O(n)`

**Danger-Inverted Prefix Sum:**

`T_total = O(n) + O(n) + O(n) = O(n)`

`Space = O(n)`

## Step 11 — Algorithm Comparison Summary

In [26]:
# ─────────────────────────────────────────────────────────────────────
# 7. Algorithm Comparison Summary
# ─────────────────────────────────────────────────────────────────────
comparison_rows = [
    [
        "Dual-Objective Sector Scoring with Weighted Prefix Draw",
        "CHOSEN",
        "O(n)",
        "O(n)",
        real_sector["Sector"],
        f"{dual_result['elapsed'] * 1000:.4f}",
        "Best balance of safety, deception, and randomness",
    ],
    [
        "Two-Phase Elimination + Weighted Draw",
        "REJECTED",
        "O(n log n)",
        "O(n)",
        two_phase_sector["Sector"],
        f"{two_phase_result['elapsed'] * 1000:.4f}",
        "Sorting cost + deterministic filtering",
    ],
    [
        "Danger-Inverted Prefix Sum",
        "REJECTED",
        "O(n)",
        "O(n)",
        danger_inverted_sector["Sector"],
        f"{danger_inverted_result['elapsed'] * 1000:.4f}",
        "Ignores Decoy_Value",
    ],
]

print("=" * 125)
print("  ALGORITHM COMPARISON")
print("=" * 125)
print(tabulate(
    comparison_rows,
    headers=["Algorithm", "Status", "Time Complexity", "Space Complexity", "Selected Sector", "Time Taken (ms)", "Main Weakness / Reason"],
    tablefmt="rounded_outline",
))

  ALGORITHM COMPARISON
╭─────────────────────────────────────────────────────────┬──────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────────────────────────────────────╮
│ Algorithm                                               │ Status   │ Time Complexity   │ Space Complexity   │ Selected Sector   │   Time Taken (ms) │ Main Weakness / Reason                            │
├─────────────────────────────────────────────────────────┼──────────┼───────────────────┼────────────────────┼───────────────────┼───────────────────┼───────────────────────────────────────────────────┤
│ Dual-Objective Sector Scoring with Weighted Prefix Draw │ CHOSEN   │ O(n)              │ O(n)               │ S4                │           19.1154 │ Best balance of safety, deception, and randomness │
│ Two-Phase Elimination + Weighted Draw                   │ REJECTED │ O(n log n)        │ O(n)               │ S2                │           11.441  │ Sorting c

## Step 12 — Rejection Analysis and Final Decision

**Two-Phase Elimination + Weighted Draw** is rejected because sorting makes it `O(n log n)`, deterministic elimination reduces unpredictability, and `Decoy_Value` is ignored.

**Danger-Inverted Prefix Sum** is rejected because it is `O(n)` and randomised, but it ignores `Decoy_Value`.

**Dual-Objective Sector Scoring with Weighted Prefix Draw** is chosen because it is `O(n)`, non-deterministic, and uses all five dataset columns.